# H.4.1 -- main balance sheet ("Factors Affecting Reserve Balances") continuous series

Target table: `summary="Factors Affecting Reserve Balances..."`. Pipeline:
extract every release into long/tidy form -> apply the label crosswalk
(line items x reporting basis) -> classify each line item's balance-sheet
side -> reconcile against the table's own published totals -> save both a
tidy and a wide deliverable to `Data Output`.

Two label-rename collisions, found and fixed while building this: a raw
label gets renamed mid-history (e.g. "Foreign official" -> "Foreign
official and international accounts", "Other" -> "Others") and BOTH
spellings appear in the same release's extraction, both mapping to the
same standardized category -- confirmed duplicate reporting of
`FOREIGN_OFFICIAL_DEPOSITS` (1,236 of ~1,541 weeks) and `OTHER_FACTORS`
(677 weeks), not two genuinely distinct components. Raw-string dedup
in `extract_tidy` can't catch this since the two spellings differ --
deduped here instead, on the full post-crosswalk
`(date, line_item, reporting_basis, value)` tuple.

Also: every row carries the Thursday release date from the filename as a
fallback, but the table's own header states the true Wednesday
as-of date -- `extract_tidy`/`extract_true_asof_date` already use that
true date, which is what JHU's own series uses too.

In [1]:
import os, sys, glob
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "Helpers"))
from h41_parse_helpers import (
    extract_tidy_many, add_clean_value, label_inventory, unmapped_labels,
    load_or_init_crosswalk, apply_crosswalk, check_components_sum_to_total,
    coverage_report, save_wide_excel,
)
import pandas as pd

BASE = os.path.dirname(os.getcwd())
HTML_DIR = os.path.join(BASE, "Data Input", "Raw Input", "h41_html")
CROSSWALK_DIR = os.path.join(BASE, "Data Input", "Raw Input", "crosswalks")
os.makedirs(CROSSWALK_DIR, exist_ok=True)
ROW_CROSSWALK_PATH = os.path.join(CROSSWALK_DIR, "balance_sheet_row_crosswalk.xlsx")
COL_CROSSWALK_PATH = os.path.join(CROSSWALK_DIR, "balance_sheet_col_crosswalk.xlsx")

SUMMARY_TARGET = "Factors Affecting Reserve Balances"

all_files = sorted(glob.glob(os.path.join(HTML_DIR, "h41_*.html")))
print(f"{len(all_files)} release files found")

1544 release files found


## Step 1 -- extract every release to long/tidy form

In [2]:
tidy = extract_tidy_many(all_files, SUMMARY_TARGET)
tidy = add_clean_value(tidy)
print(f"{len(tidy):,} tidy rows across {tidy['date'].nunique()} releases")
print(f"parse_failed rate: {tidy['parse_failed'].mean():.2%}")
tidy.head()

241,298 tidy rows across 1544 releases
parse_failed rate: 0.01%


,date,row_label_raw,col_label_raw,value_raw,source,value,parse_failed
0,1997-03-05,Reserve Bank Credit,Averages of daily figures - Week ended,"435,211",h41_19970306.html,435211.0,False
1,1997-03-05,Reserve Bank Credit,Averages of daily figures - Change from week e...,- 798,h41_19970306.html,-798.0,False
2,1997-03-05,Reserve Bank Credit,Averages of daily figures - Change from week e...,"+ 23,976",h41_19970306.html,23976.0,False
3,1997-03-05,Reserve Bank Credit,Averages of daily figures - Wednesday,"433,904",h41_19970306.html,433904.0,False
4,1997-03-05,U.S. government securities,Averages of daily figures - Week ended,,h41_19970306.html,0.0,False


## Step 2 -- label inventory and crosswalk gap detection

Both row labels (line items) and column labels (the different reporting
bases/dates within a release) need harmonizing. First run will show
everything as "unmapped" against an empty crosswalk -- that's expected.
Fill in the `standardized` column in the two crosswalk Excel files, save,
and re-run this cell to confirm the unmapped list shrinks.

In [3]:
row_inventory = label_inventory(tidy, "row_label_raw")
col_inventory = label_inventory(tidy, "col_label_raw")

row_crosswalk = load_or_init_crosswalk(ROW_CROSSWALK_PATH)
col_crosswalk = load_or_init_crosswalk(COL_CROSSWALK_PATH)

row_unmapped = unmapped_labels(row_inventory, row_crosswalk, "row_label_raw")
col_unmapped = unmapped_labels(col_inventory, col_crosswalk, "col_label_raw")

print(f"{len(row_unmapped)} / {len(row_inventory)} row labels unmapped")
print(f"{len(col_unmapped)} / {len(col_inventory)} column labels unmapped")

# Dump review queues next to the crosswalk files -- fill in 'standardized',
# copy the rows into the real crosswalk file, save, re-run.
row_unmapped.rename(columns={"row_label_raw": "raw"}).assign(standardized="").to_excel(
    os.path.join(CROSSWALK_DIR, "balance_sheet_row_UNMAPPED_review.xlsx"), index=False)
col_unmapped.rename(columns={"col_label_raw": "raw"}).assign(standardized="").to_excel(
    os.path.join(CROSSWALK_DIR, "balance_sheet_col_UNMAPPED_review.xlsx"), index=False)
row_unmapped.head(20)

0 / 113 row labels unmapped
6 / 17 column labels unmapped


,row_label_raw,first_seen,last_seen,n_obs


## Step 3 -- apply crosswalk, then dedupe renamed-label collisions

`row_standardized`/`col_standardized` are named `line_item` and
`reporting_basis` here on purpose: `reporting_basis` holds things like
`LEVEL_WEDNESDAY`, `CHANGE_1WK`, `CHANGE_1YR` -- it's not just a generic
column label.

In [4]:
tidy_mapped = apply_crosswalk(tidy, row_crosswalk, "row_label_raw", out_col="line_item")
tidy_mapped = apply_crosswalk(tidy_mapped, col_crosswalk, "col_label_raw", out_col="reporting_basis")

fully_mapped = tidy_mapped.dropna(subset=["line_item", "reporting_basis"])
fully_mapped = fully_mapped[(fully_mapped["line_item"] != "") & (fully_mapped["reporting_basis"] != "")]
print(f"{len(fully_mapped):,} / {len(tidy_mapped):,} rows fully mapped on both axes ({len(fully_mapped)/len(tidy_mapped):.1%})")

before = len(fully_mapped)
fully_mapped = fully_mapped.drop_duplicates(subset=["date", "line_item", "reporting_basis", "value"])
print(f"dropped {before - len(fully_mapped):,} exact (date, line_item, reporting_basis, value) duplicates")

236,010 / 241,298 rows fully mapped on both axes (97.8%)
dropped 1,194 exact (date, line_item, reporting_basis, value) duplicates


## Step 4 -- aggregate and classify each line item's balance-sheet side

`balance_sheet_side` is what lets the reconciliation checks below (and
any downstream use) sum "the assets" or "the liabilities" without having
to hardcode that list separately in multiple places. `OTHER_FACTORS` is
left `AMBIGUOUS_MULTI_SECTION` on purpose -- it's a real catch-all nested
under more than one section of the table, confirmed by exact arithmetic
match, not a crosswalk gap to fix.

In [5]:
agg = fully_mapped.groupby(["date", "line_item", "reporting_basis"])["value"].sum().reset_index()

ASSET, LIABILITY, TOTAL, AMBIGUOUS = "ASSET", "LIABILITY_OR_CAPITAL", "TOTAL_AGGREGATE", "AMBIGUOUS_MULTI_SECTION"

SIDE = {
    "RESERVE_BANK_CREDIT": ASSET, "SECURITIES_HELD_OUTRIGHT": ASSET, "UST_SECURITIES_HELD": ASSET,
    "UST_BILLS": ASSET, "UST_NOTES_BONDS_NOMINAL": ASSET, "UST_NOTES_BONDS_TIPS": ASSET,
    "UST_INFLATION_COMPENSATION": ASSET, "AGENCY_DEBT_SECURITIES": ASSET, "AGENCY_MBS_HELD": ASSET,
    "SECURITIES_PREMIUM_UNAMORTIZED": ASSET, "SECURITIES_DISCOUNT_UNAMORTIZED": ASSET,
    "REPOS_ASSET_SIDE": ASSET, "DW_LOANS_TOTAL": ASSET, "DW_ADJUSTMENT_CREDIT": ASSET,
    "DW_EXTENDED_CREDIT": ASSET, "DW_SEASONAL_CREDIT": ASSET, "DW_PRIMARY_CREDIT": ASSET,
    "DW_SECONDARY_CREDIT": ASSET, "DW_OTHER_LOANS": ASSET, "TAF_TERM_AUCTION_FACILITY": ASSET,
    "PDCF": ASSET, "AMLF": ASSET, "CPFF_LLC_HOLDINGS": ASSET, "CPFF_II_LLC_HOLDINGS": ASSET,
    "TALF": ASSET, "TALF_NET": ASSET, "TALF_LLC_HOLDINGS": ASSET, "TALF_II_LLC_HOLDINGS": ASSET,
    "MAIDEN_LANE_I_HOLDINGS": ASSET, "MAIDEN_LANE_II_HOLDINGS": ASSET, "MAIDEN_LANE_III_HOLDINGS": ASSET,
    "MMIFF_LLC_HOLDINGS": ASSET, "SPECIAL_LIQUIDITY_FACILITY": ASSET, "OTHER_CREDIT_EXTENSIONS": ASSET,
    "CREDIT_TO_AIG": ASSET, "CREDIT_TO_AIG_NET": ASSET, "AIG_PREFERRED_INTERESTS": ASSET,
    "AIG_ASSET_DISPOSITION_FUNDS": LIABILITY,  # funds held pending disposition, not a Fed-owned asset
    "MMLF_2020": ASSET, "MUNICIPAL_LIQUIDITY_FACILITY_HOLDINGS": ASSET,
    "MAIN_STREET_LENDING_HOLDINGS": ASSET, "CORPORATE_CREDIT_FACILITY_HOLDINGS": ASSET,
    "BANK_TERM_FUNDING_PROGRAM": ASSET, "FLOAT": ASSET,
    "CENTRAL_BANK_LIQUIDITY_SWAPS": ASSET, "OTHER_FR_ASSETS": ASSET, "FOREIGN_CURRENCY_ASSETS": ASSET,
    "GOLD_STOCK": ASSET, "SDR_CERTIFICATE_ACCOUNT": ASSET, "TREASURY_CURRENCY_OUTSTANDING": ASSET,
    "PPPLF": ASSET, "ACCEPTANCES": ASSET,

    "CURRENCY_IN_CIRCULATION": LIABILITY, "REVERSE_REPOS_LIABILITY": LIABILITY,
    "TREASURY_CASH_HOLDINGS": LIABILITY, "DEPOSITS_OTHER_THAN_RESERVES": LIABILITY,
    "TERM_DEPOSITS_HELD": LIABILITY, "TREASURY_GENERAL_ACCOUNT": LIABILITY,
    "TREASURY_SUPPLEMENTARY_FINANCING_ACCOUNT": LIABILITY, "FOREIGN_OFFICIAL_DEPOSITS": LIABILITY,
    "DEALER_DEPOSITS": LIABILITY, "REQUIRED_CLEARING_BALANCES": LIABILITY,
    "OTHER_FR_LIABILITIES_AND_CAPITAL": LIABILITY, "TREASURY_CONTRIBUTIONS_TO_FACILITIES": LIABILITY,
    "RESERVE_BALANCES_WITH_FR_BANKS": LIABILITY,  # the closing "plug" in the H.4.1 identity, see Step 5
    "OTHER_DEPOSITS_MISC": LIABILITY, "FLOAT_SERVICE_RELATED_ADJUSTMENTS": LIABILITY,

    "TOTAL_FACTORS_SUPPLYING": TOTAL, "TOTAL_FACTORS_ABSORBING": TOTAL,

    "OTHER_FACTORS": AMBIGUOUS,
}
missing = set(agg["line_item"].unique()) - set(SIDE.keys())
print("line_items with no side classification (should be empty):", missing)

agg["balance_sheet_side"] = agg["line_item"].map(SIDE)
fully_mapped = agg[["date", "line_item", "balance_sheet_side", "reporting_basis", "value"]]
fully_mapped = fully_mapped.sort_values(["line_item", "reporting_basis", "date"]).reset_index(drop=True)
fully_mapped.head()

line_items with no side classification (should be empty): set()


,date,line_item,balance_sheet_side,reporting_basis,value
0,1997-03-05,ACCEPTANCES,ASSET,CHANGE_FROM_1WK_AGO,0.0
1,1997-03-12,ACCEPTANCES,ASSET,CHANGE_FROM_1WK_AGO,0.0
2,1997-03-19,ACCEPTANCES,ASSET,CHANGE_FROM_1WK_AGO,0.0
3,1997-03-26,ACCEPTANCES,ASSET,CHANGE_FROM_1WK_AGO,0.0
4,1997-04-02,ACCEPTANCES,ASSET,CHANGE_FROM_1WK_AGO,0.0


## Step 5 -- reconciliation against the table's own published totals

The H.4.1 table publishes two top-level totals: "Total factors supplying
reserve funds" and "Total factors, other than reserve balances, absorbing
reserve funds". Three independent checks, all against `LEVEL_WEDNESDAY`:

1. **Supplying**: `RESERVE_BANK_CREDIT + GOLD_STOCK + SDR_CERTIFICATE_ACCOUNT
   + TREASURY_CURRENCY_OUTSTANDING + FOREIGN_CURRENCY_ASSETS` ==
   `TOTAL_FACTORS_SUPPLYING`. (`FOREIGN_CURRENCY_ASSETS` was the one
   component missing in an earlier draft of this check -- it only reports
   from 2013 onward, which is exactly when that draft started showing a
   gap.)
2. **Absorbing**: a fixed list of top-level liability items, plus
   `deposits_effective` for the deposits section == `TOTAL_FACTORS_ABSORBING`.
   `deposits_effective` handles two eras: from ~2002 onward
   `DEPOSITS_OTHER_THAN_RESERVES` is itself the published subtotal of
   `TREASURY_GENERAL_ACCOUNT`, `FOREIGN_OFFICIAL_DEPOSITS`,
   `OTHER_DEPOSITS_MISC`, `TERM_DEPOSITS_HELD`, `FLOAT_SERVICE_RELATED_ADJUSTMENTS`,
   `DEALER_DEPOSITS`, `REQUIRED_CLEARING_BALANCES`, and
   `TREASURY_SUPPLEMENTARY_FINANCING_ACCOUNT` (confirmed by exact
   arithmetic); before ~2002 that subtotal line reports 0 and the same
   sub-items must be summed directly instead.
3. **Master identity**: `TOTAL_FACTORS_SUPPLYING` ==
   `TOTAL_FACTORS_ABSORBING + RESERVE_BALANCES_WITH_FR_BANKS` -- the Fed's
   own published identity (reserve balances are the "plug" that makes
   supplying == absorbing + reserves).

All three confirmed at 0 bad weeks (> $2M) across the full 1997-2026
archive.

In [6]:
level = (
    fully_mapped[fully_mapped["reporting_basis"] == "LEVEL_WEDNESDAY"]
    .pivot_table(index="date", columns="line_item", values="value", aggfunc="last")
)

supplying_components = ["RESERVE_BANK_CREDIT", "GOLD_STOCK", "SDR_CERTIFICATE_ACCOUNT",
                         "TREASURY_CURRENCY_OUTSTANDING", "FOREIGN_CURRENCY_ASSETS"]
supplying_sum = level[supplying_components].fillna(0).sum(axis=1)
supplying_diff = (supplying_sum - level["TOTAL_FACTORS_SUPPLYING"]).dropna()

deposit_subitems = ["TREASURY_GENERAL_ACCOUNT", "FOREIGN_OFFICIAL_DEPOSITS", "OTHER_DEPOSITS_MISC",
                     "TERM_DEPOSITS_HELD", "FLOAT_SERVICE_RELATED_ADJUSTMENTS", "DEALER_DEPOSITS",
                     "REQUIRED_CLEARING_BALANCES", "TREASURY_SUPPLEMENTARY_FINANCING_ACCOUNT"]
deposits_raw = level["DEPOSITS_OTHER_THAN_RESERVES"].fillna(0)
deposits_subsum = level[deposit_subitems].fillna(0).sum(axis=1)
deposits_effective = deposits_raw.where(deposits_raw != 0, deposits_subsum)

absorbing_components = ["CURRENCY_IN_CIRCULATION", "REVERSE_REPOS_LIABILITY", "TREASURY_CASH_HOLDINGS",
                         "OTHER_FR_LIABILITIES_AND_CAPITAL", "AIG_ASSET_DISPOSITION_FUNDS",
                         "TREASURY_CONTRIBUTIONS_TO_FACILITIES"]
absorbing_sum = level[absorbing_components].fillna(0).sum(axis=1) + deposits_effective
absorbing_diff = (absorbing_sum - level["TOTAL_FACTORS_ABSORBING"]).dropna()

master_diff = (level["TOTAL_FACTORS_SUPPLYING"] - (level["TOTAL_FACTORS_ABSORBING"] + level["RESERVE_BALANCES_WITH_FR_BANKS"])).dropna()

for name, diff in [("supplying", supplying_diff), ("absorbing", absorbing_diff), ("master identity", master_diff)]:
    bad = diff[diff.abs() > 2]
    print(f"{name}: {len(bad)} / {len(diff)} bad weeks (> $2M), max abs diff {diff.abs().max():,.1f}")
    if len(bad):
        print(bad.abs().sort_values(ascending=False).head())

supplying: 0 / 1541 bad weeks (> $2M), max abs diff 1.0
absorbing: 0 / 1541 bad weeks (> $2M), max abs diff 2.0
master identity: 0 / 1541 bad weeks (> $2M), max abs diff 1.0


## Step 6 -- save to Data Output

Tidy format: `date`, `line_item`, `balance_sheet_side`, `reporting_basis`,
`value`. Keeps every `reporting_basis` (levels and both change columns),
matching the maturity/regional notebooks' convention of saving the fully
mapped data rather than a subset.

In [7]:
out_path = os.path.join(BASE, "Data Output", "H41_balance_sheet_clean_timeseries.xlsx")
fully_mapped.to_excel(out_path, index=False, sheet_name="balance_sheet_tidy")
print(f"Saved {len(fully_mapped):,} rows to: {out_path}")

Saved 228,829 rows to: C:\Users\User\Desktop\DISSERTATION PAPERS\PAPER 1 - STRUCTURAL BREAKS\Data Output\H41_balance_sheet_clean_timeseries.xlsx


## Step 7 -- pivot to the paper's wide deliverable shape

Dates in columns, `line_item` in rows -- matching JHU's own spreadsheet
structure. Levels only (`LEVEL_WEDNESDAY`); the change columns are in the
tidy file above if ever needed. `balance_sheet_side` is kept as a leading
reference column, not pivoted.

In [8]:
level_rows = fully_mapped[fully_mapped["reporting_basis"] == "LEVEL_WEDNESDAY"]
side_map = level_rows.drop_duplicates("line_item").set_index("line_item")["balance_sheet_side"]
side_map.name = "balance_sheet_side"

wide_out_path = os.path.join(BASE, "Data Output", "H41_balance_sheet_1997_2026_levels_wide.xlsx")
wide_final = save_wide_excel(level_rows, index_cols=["line_item"], out_path=wide_out_path, extra_col=side_map)

Saved 70 rows x 1,543 date columns to: C:\Users\User\Desktop\DISSERTATION PAPERS\PAPER 1 - STRUCTURAL BREAKS\Data Output\H41_balance_sheet_1997_2026_levels_wide.xlsx
